# EDA | Playing Cards Classification (53 classes)

**Mục tiêu:** Kiểm tra phân bố lớp, chất lượng ảnh, đặc trưng hình ảnh, trùng lặp và độ tin cậy của phép chia train/val/test. Notebook **chỉ phân tích**, không sửa dữ liệu và không chia lại tập. Mỗi mục có nhận xét tự động dựa trên số liệu thực tế.

**Cấu trúc:** `notebooks/EDA.ipynb` · `data/raw/dataset.zip` · `data/metadata/{clean_metadata.csv,label_map.json,train.csv,val.csv,test.csv}`.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
from collections import Counter
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
from IPython.display import display, Markdown

# Works when notebook is launched from notebooks/ or project root.
ROOT = Path.cwd().resolve()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data").is_dir():
    raise FileNotFoundError("Run this notebook from the project root or notebooks/ directory")

RAW_DIR = ROOT / "data" / "raw"
METADATA_DIR = ROOT / "data" / "metadata"
ARCHIVE = RAW_DIR / "dataset.zip"
CLEAN_METADATA = METADATA_DIR / "clean_metadata.csv"
LABEL_MAP = METADATA_DIR / "label_map.json"
SPLIT_FILES = {s: METADATA_DIR / f"{s}.csv" for s in ("train", "val", "test")}

NUM_CLASSES = 53
SEED = 42
SAMPLE_FOR_PIXELS = 600
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for path in [ARCHIVE, CLEAN_METADATA, LABEL_MAP, *SPLIT_FILES.values()]:
    print(f"{'OK' if path.is_file() else 'MISSING':7} {path.relative_to(ROOT)}")
if not ARCHIVE.is_file():
    raise FileNotFoundError(ARCHIVE)

## 1. Danh mục ảnh và nhãn

Đọc danh sách file từ ZIP. Ưu tiên `clean_metadata.csv` để lấy nhãn; nếu thiếu nhãn mới suy luận từ thư mục cha. Giữ `member` là **đường dẫn bên trong ZIP**, không phải đường dẫn trên ổ đĩa.

In [ ]:
def normalized_path(s):
    return str(s).replace("\\", "/").removeprefix("./").strip("/")

SKIP_DIRS = {"train", "training", "val", "valid", "validation", "test", "testing", "images", "image", "dataset", "dataset_cleaned", "cards"}

def inferred_class(member):
    parts = Path(member).parts[:-1]
    options = [p for p in parts if p.lower() not in SKIP_DIRS and not p.startswith("__MACOSX")]
    return options[-1] if options else None

def inferred_split(member):
    aliases = {"train": "train", "training": "train", "val": "val", "valid": "val", "validation": "val", "test": "test", "testing": "test"}
    return next((aliases[p.lower()] for p in Path(member).parts[:-1] if p.lower() in aliases), None)

with ZipFile(ARCHIVE) as z:
    members = sorted(m for m in z.namelist() if not m.endswith("/") and Path(m).suffix.lower() in IMAGE_EXTS and "__MACOSX" not in Path(m).parts)

if not members:
    raise ValueError("ZIP contains no supported image files")
df = pd.DataFrame({"member": members})
df["extension"] = df.member.map(lambda m: Path(m).suffix.lower())
df["archive_split"] = df.member.map(inferred_split)
df["class_name"] = df.member.map(inferred_class)

if CLEAN_METADATA.is_file():
    meta = pd.read_csv(CLEAN_METADATA)
    print("clean_metadata.csv columns:", list(meta.columns))
    if {"member", "class_name"}.issubset(meta.columns):
        meta = meta.dropna(subset=["member"]).copy()
        meta["member"] = meta.member.map(normalized_path)
        duplicates = meta.groupby("member")["class_name"].nunique().gt(1).sum()
        print("Conflicting class labels in metadata:", duplicates)
        label_lookup = meta.drop_duplicates("member").set_index("member")["class_name"]
        df["class_name"] = df.member.map(label_lookup).fillna(df.class_name)
        print("Images matched to clean metadata:", df.member.isin(label_lookup.index).sum(), "/", len(df))
    else:
        print("Metadata lacks member/class_name; using folder labels")

print(f"Images: {len(df):,} | Classes: {df.class_name.nunique()} | Missing labels: {df.class_name.isna().sum()}")
print("File formats:", df.extension.value_counts().to_dict())
display(df.head())
if df.class_name.isna().any():
    display(df[df.class_name.isna()].head(10))
if df.class_name.nunique() != NUM_CLASSES:
    print(f"NOTICE: Expected {NUM_CLASSES} classes; found {df.class_name.nunique()}.")

## 2. Phân bố lớp

Đánh giá số lượng ảnh ở mỗi lớp và tỷ lệ mất cân bằng. Chênh lệch quá lớn làm độ chính xác chung kém đại diện cho lớp ít mẫu.

In [ ]:
counts = df.class_name.value_counts().sort_values()
class_stats = pd.DataFrame({"images": counts, "share_%": (counts / len(df) * 100).round(2)})
display(class_stats.sort_values("images", ascending=False))
fig, ax = plt.subplots(figsize=(15, 5))
counts.plot.bar(ax=ax)
ax.set(title="Images per class", xlabel="Class", ylabel="Image count")
plt.xticks(rotation=90, fontsize=8)
plt.tight_layout()
plt.show()
if len(counts):
    ratio = counts.max() / max(counts.min(), 1)
    print(f"Range: {counts.min()}–{counts.max()} images/class | Max/min ratio: {ratio:.2f}")
    print("Nhận xét:", "Chênh lệch số mẫu đáng chú ý; cần báo cáo macro-F1 và recall từng lớp." if ratio > 1.5 else "Phân bố lớp tương đối đồng đều theo số lượng ảnh.")

## 3. Chất lượng và kích thước ảnh

Kiểm tra ảnh đọc được hay không, độ phân giải, tỷ lệ chiều rộng/chiều cao và số kênh. Một ảnh có thể nằm trong ZIP nhưng bị hỏng hoặc không chuyển đổi được sang RGB.

In [ ]:
records = []
with ZipFile(ARCHIVE) as z:
    for member in df.member:
        try:
            with Image.open(BytesIO(z.read(member))) as im:
                im.load()
                width, height, mode = im.width, im.height, im.mode
            records.append((member, width, height, mode, None))
        except Exception as exc:
            records.append((member, np.nan, np.nan, None, type(exc).__name__))
quality = pd.DataFrame(records, columns=["member", "width", "height", "mode", "error"])
df = df.merge(quality, on="member", how="left", validate="one_to_one")
valid = df[df.error.isna()].copy()
valid["aspect_ratio"] = valid.width / valid.height
valid["pixels"] = valid.width * valid.height
print(f"Readable: {len(valid):,}/{len(df):,} | Invalid: {df.error.notna().sum()}")
display(valid[["width", "height", "aspect_ratio", "pixels"]].describe().round(2))
print("Color modes:", valid["mode"].value_counts().to_dict())
if df.error.notna().any():
    display(df.loc[df.error.notna(), ["member", "error"]].head(15))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(valid.width, bins=30)
axes[0].set(title="Image width", xlabel="Pixels", ylabel="Count")
axes[1].hist(valid.aspect_ratio, bins=30)
axes[1].set(title="Aspect ratio (width/height)", xlabel="Ratio", ylabel="Count")
plt.tight_layout()
plt.show()
size_variants = valid.groupby(["width", "height"]).size().sort_values(ascending=False)
print("Most common resolutions:")
display(size_variants.head(10).rename("images").to_frame())
print("Nhận xét:", "Ảnh có nhiều kích thước/tỷ lệ; cần resize nhất quán khi training." if len(size_variants) > 1 else "Ảnh có cùng độ phân giải; bước resize vẫn phải thống nhất giữa các mô hình.")

## 4. Ảnh mẫu theo lớp

Quan sát màu nền, vị trí lá bài, góc xoay, độ sắc nét và những lớp dễ nhầm. Hiển thị một ảnh ngẫu nhiên của tối đa 18 lớp để notebook không quá dài.

In [ ]:
rng = np.random.default_rng(SEED)
classes = sorted(valid.class_name.dropna().unique())
selected_classes = rng.choice(classes, size=min(18, len(classes)), replace=False) if classes else []
fig, axes = plt.subplots(3, 6, figsize=(15, 9))
with ZipFile(ARCHIVE) as z:
    for ax, cls in zip(axes.flat, selected_classes):
        member = valid.loc[valid.class_name == cls, "member"].sample(1, random_state=SEED).iloc[0]
        with Image.open(BytesIO(z.read(member))) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(str(cls), fontsize=9)
    for ax in axes.flat:
        ax.axis("off")
plt.tight_layout()
plt.show()
print("Nhận xét: Quan sát trực quan giúp xác định nền ảnh, góc đặt bài và mức đa dạng trong từng lớp; đây là đánh giá định tính, không suy ra độ chính xác mô hình.")

## 5. Màu sắc, độ sáng và tương phản

Lấy mẫu có giới hạn để tránh chi phí đọc mọi pixel. Thống kê RGB trên ảnh chuyển sang RGB và resize 64×64; độ sáng và tương phản được tính từ ảnh xám.

In [ ]:
sample = valid.sample(n=min(SAMPLE_FOR_PIXELS, len(valid)), random_state=SEED)
colors, brightness, contrast = [], [], []
with ZipFile(ARCHIVE) as z:
    for member in sample.member:
        with Image.open(BytesIO(z.read(member))) as im:
            rgb = np.asarray(im.convert("RGB").resize((64, 64)), dtype=np.float32) / 255.0
        colors.append(rgb.mean(axis=(0, 1)))
        gray = rgb @ np.array([0.299, 0.587, 0.114])
        brightness.append(gray.mean())
        contrast.append(gray.std())
colors = np.asarray(colors)
pixel_stats = pd.DataFrame(colors, columns=["R", "G", "B"])
pixel_stats["brightness"] = brightness
pixel_stats["contrast"] = contrast
display(pixel_stats.describe().round(3))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for channel in ("R", "G", "B"):
    axes[0].hist(pixel_stats[channel], bins=25, alpha=.45, label=channel)
axes[0].set(title="Mean RGB per image", xlabel="Normalized channel mean")
axes[0].legend()
axes[1].scatter(brightness, contrast, s=12, alpha=.5)
axes[1].set(title="Brightness vs contrast", xlabel="Brightness", ylabel="Contrast")
plt.tight_layout()
plt.show()
print(f"Nhận xét: Độ sáng trung bình {np.mean(brightness):.3f}, độ lệch chuẩn giữa các ảnh {np.std(brightness):.3f}. Thống kê mô tả mẫu {len(sample)} ảnh, không phải toàn bộ dataset.")

## 6. Ảnh trùng chính xác và nhãn xung đột

SHA-256 tìm file **giống hệt về byte**. Hai ảnh trông giống nhau nhưng khác nén/cắt/xoay có thể không cùng hash; vì vậy bước này **không** phát hiện near-duplicates.

In [ ]:
hashes = {}
with ZipFile(ARCHIVE) as z:
    for member in df.member:
        hashes[member] = hashlib.sha256(z.read(member)).hexdigest()
df["sha256"] = df.member.map(hashes)
dup_groups = df.groupby("sha256").size()
duplicate_hashes = dup_groups[dup_groups > 1].index
duplicate_rows = df[df.sha256.isin(duplicate_hashes)].sort_values("sha256")
conflicting_hashes = df.groupby("sha256").class_name.nunique(dropna=True)
conflicting_hashes = conflicting_hashes[conflicting_hashes > 1].index
print("Exact duplicate groups:", len(duplicate_hashes))
print("Extra duplicate files:", int((dup_groups - 1).clip(lower=0).sum()))
print("Hashes with conflicting labels:", len(conflicting_hashes))
if not duplicate_rows.empty:
    display(duplicate_rows[["member", "class_name", "sha256"]].head(20))
print("Nhận xét:", "Có file trùng byte; cần kiểm tra sự phân bổ vào các tập trước khi training." if len(duplicate_hashes) else "Không thấy ảnh trùng byte; chưa thể loại trừ ảnh gần trùng.")

## 7. Kiểm tra train / validation / test

Đọc **đúng** ba CSV split. Kiểm tra tên ảnh khớp ZIP, ảnh nằm ở nhiều split, phân bố lớp từng split và hash trùng giữa các split. Không tự động tạo split mới hay xóa ảnh.

In [ ]:
parts = []
for split_name, path in SPLIT_FILES.items():
    if not path.is_file():
        print("Missing split file:", path.name)
        continue
    part = pd.read_csv(path)
    print(path.name, "rows:", len(part), "columns:", list(part.columns))
    if "member" not in part:
        print("  Skipped: member column not found")
        continue
    p = part[["member"]].dropna().copy()
    p["member"] = p.member.map(normalized_path)
    p["split"] = split_name
    parts.append(p)

split_df = df[["member", "class_name", "sha256", "archive_split"]].copy()
split_df["split"] = split_df.archive_split
if parts:
    assignments = pd.concat(parts, ignore_index=True)
    conflicting_members = assignments.groupby("member").split.nunique()
    print("Members assigned to multiple splits:", int((conflicting_members > 1).sum()))
    missing_members = set(assignments.member) - set(df.member)
    print("CSV members missing from ZIP:", len(missing_members))
    print("Duplicate member rows in split files:", int(assignments.duplicated(["member", "split"]).sum()))
    assigned = assignments.drop_duplicates("member").set_index("member").split
    split_df["split"] = split_df.member.map(assigned).fillna(split_df.archive_split)
    print("ZIP members without CSV assignment:", int((~df.member.isin(assigned.index)).sum()))
    if missing_members:
        print("Example missing paths:", sorted(missing_members)[:5])
else:
    print("No usable CSV splits: using folder information if available")

known = split_df[split_df.split.isin(["train", "val", "test"])].copy()
if known.empty:
    print("No usable split assignments; cannot audit cross-split leakage")
else:
    summary = known.split.value_counts().reindex(["train", "val", "test"], fill_value=0)
    display(pd.DataFrame({"images": summary, "share_%": (summary / len(known) * 100).round(2)}))
    class_table = pd.crosstab(known.class_name, known.split).reindex(columns=["train", "val", "test"], fill_value=0)
    display(class_table)
    print("Classes missing in at least one split:", int((class_table == 0).any(axis=1).sum()))
    ax = class_table.plot.bar(figsize=(15, 5), width=.8, title="Class distribution by split")
    ax.set(xlabel="Class", ylabel="Number of images")
    plt.xticks(rotation=90, fontsize=8)
    plt.tight_layout()
    plt.show()
    cross_hash = known.groupby("sha256").split.nunique()
    leaked_hashes = cross_hash[cross_hash > 1].index
    print("Exact duplicate hashes across splits:", len(leaked_hashes))
    if len(leaked_hashes):
        display(known[known.sha256.isin(leaked_hashes)].sort_values("sha256").head(20))
    print("Nhận xét:", "Có exact data leakage giữa các split; metric test/validation có nguy cơ bị thổi phồng." if len(leaked_hashes) else "Không thấy trùng byte giữa các split đã gán; near-duplicate leakage vẫn chưa được kiểm tra.")

## 8. Tổng kết EDA

Kết luận được tạo từ kết quả thực tế của các cell trước; phân biệt rõ vấn đề đã xác minh và vấn đề cần kiểm tra thêm. Không kết luận mô hình overfit hay đạt hiệu năng tốt chỉ từ EDA.

In [ ]:
notes = [
    f"Dataset: **{len(df):,} file ảnh**, **{df.class_name.nunique()} lớp** (kỳ vọng {NUM_CLASSES}).",
    f"Chất lượng file: **{df.error.notna().sum()}** file ảnh không đọc được; **{len(size_variants)}** độ phân giải khác nhau.",
    f"Cân bằng lớp: ít nhất **{counts.min()}**, nhiều nhất **{counts.max()}** ảnh/lớp; tỷ lệ max/min **{counts.max()/max(counts.min(), 1):.2f}**.",
    f"Trùng byte: **{len(duplicate_hashes)}** nhóm SHA-256 trùng; **{len(conflicting_hashes)}** hash có nhãn không nhất quán.",
]
if known.empty:
    notes.append("Split: chưa có thông tin split hợp lệ để xác nhận data leakage.")
else:
    notes.append(f"Split: **{len(known):,}** ảnh có nhãn split; **{len(leaked_hashes)}** hash trùng giữa các split.")
notes.append("Giới hạn: SHA-256 không phát hiện near-duplicates; phân tích RGB dựa trên mẫu ảnh chứ không phải toàn bộ pixel dataset.")
display(Markdown("### Tóm tắt kết quả\n" + "\n".join(f"- {note}" for note in notes)))